# AI Enabled Visualization

## Prerequisites

We assume you have access to both Azure OpenAI and have already deployed an LLM.

## Get started

In this tutorial we will be using Azure OpenAI which (if you havent already deployed) you can learn how to deploy [here](https://github.com/STRIDES/NIHCloudLabAzure/blob/main/notebooks/GenAI/azure_infra_setup/README.md). This tutorial utilizes the model gpt-5.4-nano and the embeddings model text-embedding-3-small.

## Setting up Python and Azure OpenAI environment

In [ ]:
# azure cloud login
!az login --identity

In [ ]:
# set your subscription
!az account set --subscription "NIH.CIT.CS.CloudLab.Azure_XXX" # your subscription name

In [ ]:
# confirm subscription
!az account show

In [ ]:
%pip install --quiet openai

### Get Azure OpenAI environment from Azure
In search bar window, type **Foundry** and select **Microsoft Foundry**

In microsoft foundry window, click on **resource** where you deployed gpt and embedding models

For Azure OpenAI endpoint, click rectangle under **Azure OpenAI endpoint**. It would copy OpenAI endpoint, replace it in below cell with **your_openai_endpoint**

For Azure OpenAI Key, click rectangle under **API key**. It would copy OpenAI key, replace it in below cell with **yourOpenAIKey**

In [ ]:
# configure environment
import os, time, json, numpy as np, pandas as pd
from typing import List, Dict
from xml.etree import ElementTree as ET
from openai import OpenAI
import requests

AZURE_OPENAI_ENDPOINT="yourOpenAIEndpoint"
AZURE_OPENAI_KEY = "yourOpenAIKey"
AZURE_OPENAI_DEPLOYMENT_EMBED =  "text-embedding-3-small"
AZURE_OPENAI_DEPLOYMENT_CHAT =  "gpt-5.4-nano"

client = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_KEY)

### Load your data and results

In [ ]:
import pandas as pd

# Load gene X samples data 
expr = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/DESeq2_py_gene_samples.csv", index_col=0)
expr.head()


In [ ]:
# Load metadata 
meta = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/samples_labels.csv")
print("Sample metadata:", meta.shape)
meta.head()

### Shrinkage in DESeq2  
shrinkage of log₂ fold changes (LFC) is a post‑analysis step that adjusts raw LFC estimates to make them more reliable, especially for genes with low counts or high dispersion. 
It is recommended for most RNA‑seq datasets, as it improves comparability across experiments and stability in downstream analysis.

In [ ]:
# Load DESeq2 result tables
res_shrunken = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/DESEQ2_py_results_shrunken.csv")
res_full = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/DESEQ2_py_results.csv")

print("Shrunken results:", res_shrunken.shape)
print(res_shrunken.head())

print("\nFull results:", res_full.shape)
print(res_full.head())

### Top 50 most significant up and down regulated Genes

In [ ]:
# Load DEG subsets
top50_down = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/top50_downregulated.csv")
top50_up = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/top50_upregulated.csv")

print("\nTop 50 downregulated:", top50_down.shape)
print(top50_down.head())

print("\nTop 50 upregulated:", top50_up.shape)
print(top50_up.head())


### Generating dataframe for PCA plot
Principal Component Analysis (PCA) is a dimensionality reduction technique used in RNA-Seq to visualize sample relationships and identify major sources of variation in gene expression data.

In [ ]:
import numpy as np

expr_t = expr.T
# --- Align expression rows to metadata order ---
meta["ID"] = meta["ID"].astype(str).str.strip()
expr_t.index = expr_t.index.astype(str).str.strip()

# Keep only samples present in both
common_ids = meta["ID"].isin(expr_t.index)

meta = meta.loc[common_ids].copy()
expr_t = expr_t.loc[meta["ID"]].copy()


In [ ]:

# Convert everything to numeric (coercing errors to NaN)
expr_t = expr_t.apply(pd.to_numeric, errors='coerce')


In [ ]:
# 1) Log2(count+1) transform for count data (comment out if not needed)
if (expr_t.values >= 0).all():  # simple guard for log transform
    expr_t = np.log2(expr_t + 1)

# 2) Z-score genes (mean=0, std=1), common for PCA on expression
expr_t_z = (expr_t - expr_t.mean(axis=0)) / expr_t.std(axis=0).replace(0, np.nan)
expr_t_z = expr_t_z.fillna(0.0)  # handle zero-variance genes safely

print("Expression (z-scored) shape:", expr_t_z.shape)
print("First rows:\n", expr_t_z.head())


### Generate PCA coordinates

In [ ]:

from sklearn.decomposition import PCA

# --- PCA on samples × genes ---
pca = PCA(n_components=2, random_state=0)
pcs = pca.fit_transform(expr_t_z.values)  # shape: (n_samples, 2)

# --- Build PCA table for Azure AI prompt (PC1/PC2 + Treatment) ---
pca_df = pd.DataFrame({
    "Sample": expr_t_z.index,
    "PC1": pcs[:, 0],
    "PC2": pcs[:, 1],
    "Treatment": meta.set_index("ID").loc[expr_t_z.index, "Treatment"].values
})

# Explained variance ratios (for labeling axes if you later plot)
var_pc1 = pca.explained_variance_ratio_[0]
var_pc2 = pca.explained_variance_ratio_[1]
print(f"Explained variance: PC1={var_pc1:.2%}, PC2={var_pc2:.2%}")

#  Save 
pca_df.to_csv("results/pca_coordinates.csv", index=False)
print(pca_df.head())


### Generate Euclidian distance for samples
Sample distance plots are used to assess overall similarity between samples: which samples are similar to each other, which are different? We calculate the euclidean distance between samples using dist () on the rlog-transformed data to ensure roughly equal

In [ ]:
from scipy.spatial.distance import pdist, squareform

# --- Euclidean distance matrix across samples ---
D_euclid = squareform(pdist(expr_t_z.values, metric="euclidean"))
euclid_df = pd.DataFrame(D_euclid, index=expr_t_z.index, columns=expr_t_z.index)
#euclid_df.to_csv("results/sample_distance_euclidean.csv")
print(euclid_df.head())

### Correlation matrix for 
In RNA-Seq data analysis, generating a correlation matrix for your samples can help you assess sample similarity, detect outliers, and validate clustering results. 

In [ ]:
# --- Correlation distance matrix across samples ---
# Pearson correlation: corr(s_i, s_j) in [-1,1]
# Convert to a distance: d_corr = 1 - corr
corr = np.corrcoef(expr_t_z.values)                     # shape (n_samples, n_samples)
D_corr = 1.0 - corr

corr_df = pd.DataFrame(D_corr, index=expr_t_z.index, columns=expr_t_z.index)
#corr_df.to_csv("results/sample_distance_correlation.csv")
print(corr_df.head())

### Spearman correlation distance for samples
Spearman correlation is a widely used statistical measure in RNA-Seq analysis to assess the monotonic relationship between expression profiles of different samples. It is particularly suitable for RNA-Seq because the method is non-parametric and does not assume normality of gene expression counts, which are often skewed or zero-inflated.

In [ ]:
# ----Spearman correlation distance ---
# Useful if ranks (monotonic relationships) are preferred

# Step 1: Rank-transform each gene column across samples
expr_rank = expr_t_z.T.rank(axis=0)

# Step 2: Sample–sample Spearman correlation
corr_spearman = expr_rank.corr(method="pearson")

# Step 3: Turn correlation into distance
D_spearman = 1 - corr_spearman

#D_spearman.to_csv("results/sample_distance_spearman.csv")
dist_spearman_df = pd.DataFrame(D_spearman)

In [ ]:
gene_samples_df = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/DESeq2_py_gene_samples.csv", index_col=0)
pca_df = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/pca_coordinates.csv")
dist_corr_df = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/sample_distance_correlation.csv")
euclid_df = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/sample_distance_euclidean.csv")
dist_spearman_df = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/sample_distance_spearman.csv")
samples_meta = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/results/samples_labels.csv")

In [ ]:
gene_samples_df.head

### Volcano plot (DESeq2 shrunken)
A volcano plot is a common visualization in RNA-Seq analysis that displays the log2 fold change against the -log10 p-value, allowing for easy identification of significantly up- or down-regulated genes.

In [ ]:

system_msg_volcano = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs. Each item MUST include:
- title (str)
- description (str)
- dependencies (list[str])
- code (str)  ← runnable Python using seaborn/matplotlib ONLY

APPROVED DATAFRAME NAMES (already loaded in the notebook):
- res_shrunken

PALETTE / STYLE CONSTRAINTS:
- Do NOT reference palette variables (palette, colorblind, colorvblind, colourblind, cb_palette) unless you DEFINE them INSIDE the code.
- If a palette is needed, INSIDE the code do ONE of:
  a) sns.set_palette("colorblind")
  b) palette = sns.color_palette("colorblind"); sns.scatterplot(..., palette=palette)

COLUMN NAMES — DO NOT INVENT:
- res_shrunken: ["Gene","baseMean","log2FoldChange","lfcSE","stat","pvalue","padj"]

ROBUSTNESS:
- Coerce numerics via pd.to_numeric(..., errors="coerce"); replace inf with NaN; drop rows with missing required fields before plotting.

FIGURE:
- Volcano plot: X=log2FoldChange, Y=−log10(padj); highlight padj<0.05; title and axis labels required.

EXPORT:
- Save figures/<safe_title>.png (dpi=300, bbox_inches="tight").

OUTPUT FORMAT:
- Return STRICT JSON ONLY (no prose, no markdown, no backticks).
"""


In [ ]:

schema_volcano = """
{
  "datasets": {
    "res_shrunken": {
      "description": "DESeq2 shrunken results",
      "columns": ["Gene","baseMean","log2FoldChange","padj"]
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""

user_msg_volcano = f"""
SCHEMA:
{schema_volcano}

GOAL:
- Generate a Volcano plot spec (STRICT JSON) using res_shrunken:
  X=log2FoldChange, Y=−log10(padj), highlight padj<0.05.

RESTRICTIONS:
- Use ONLY DataFrame name: res_shrunken.
- Define any helper variables INSIDE your code block.
- Do NOT invent column names; use only those in SCHEMA.

FORMAT:
- Return STRICT JSON array: [{{"title","description","dependencies","code"}}, ...].
"""


In [ ]:

# === VOLCANO: execute plan and plot ===
import json, os
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

# 1) Call Azure AI
resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_volcano},
        {"role": "user",   "content": user_msg_volcano}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content

# 2) Parse STRICT JSON
try:
    plan_volcano = json.loads(response_text)
except json.JSONDecodeError:
    print("Model did not return STRICT JSON. Raw response:\n", response_text)
    raise

# 3) Guardrails & environment
os.makedirs("figures", exist_ok=True)
sns.set_context("talk"); sns.set_palette("colorblind")

# aliases for hallucinated names
local_env = {
    "res_shrunken": res_shrunken,
    "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os
}
local_env.update({
    "palette": "colorblind", "colorblind": "colorblind",
    "colorvblind": "colorblind", "colourblind": "colorblind", "cb_palette": "colorblind"
})

# 4) Execute plotting code
for fig in plan_volcano:
    code = fig["code"]
    # save spec for traceability
    safe_title = fig.get("title","volcano").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig, f, indent=2)
    exec(code, {}, local_env)

print("Volcano figure(s) saved to ./figures")


### MA plot (DESeq2 shrunken)
An MA plot is a widely used visualization technique in RNA-Seq (and other high-throughput sequencing) analyses to assess differential expression and detect systematic biases in the data.

In [ ]:

system_msg_ma = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs (title, description, dependencies, code).

APPROVED DATAFRAME NAME:
- res_shrunken

PALETTE / STYLE:
- No external palette variables; define inside the code if needed (see volcano message).

COLUMNS:
- res_shrunken: ["Gene","baseMean","log2FoldChange","padj"]

ROBUSTNESS:
- Coerce numeric cols; replace inf with NaN; drop missing rows before plotting.

FIGURE:
- MA plot: X=log10(baseMean+1), Y=log2FoldChange; color points with padj<0.05; add y=0 line.

EXPORT:
- figures/<safe_title>.png (dpi=300, bbox_inches="tight")

OUTPUT:
- STRICT JSON ONLY.
"""


In [ ]:

schema_ma = """
{
  "datasets": {
    "res_shrunken": {
      "description": "DESeq2 shrunken results",
      "columns": ["Gene","baseMean","log2FoldChange","padj"]
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""

user_msg_ma = f"""
SCHEMA:
{schema_ma}

GOAL:
- Generate an MA plot spec (STRICT JSON) from res_shrunken:
  X=log10(baseMean+1), Y=log2FoldChange; highlight padj<0.05; draw horizontal y=0.

RESTRICTIONS:
- Use ONLY res_shrunken; define helper variables INSIDE code if needed.
- Do NOT invent column names.

FORMAT:
- Return STRICT JSON array of figure specs: [{{title,description,dependencies,code}}, ...].
"""


In [ ]:

# === MA PLOT: execute plan and plot ===
import json, os
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_ma},
        {"role": "user",   "content": user_msg_ma}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content
plan_ma = json.loads(response_text)

os.makedirs("figures", exist_ok=True)
sns.set_context("talk"); sns.set_palette("colorblind")

local_env = {"res_shrunken": res_shrunken, "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os}
local_env.update({"palette":"colorblind","colorblind":"colorblind","colorvblind":"colorblind","colourblind":"colorblind","cb_palette":"colorblind"})

for fig in plan_ma:
    code = fig["code"]
    safe_title = fig.get("title","MA_plot").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig,f,indent=2)
    exec(code, {}, local_env)


### Top‑50 upregulated genes barplot

In [ ]:

system_msg_topup = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs (title, description, dependencies, code).

APPROVED DATAFRAME NAME:
- top50_up

PALETTE / STYLE:
- No external palette variables; define inside code if needed.

COLUMNS:
- top50_up: ["Gene","log2FoldChange"]  (optional "padj")

FIGURE:
- Barplot/strip for Top‑50 Up: x=gene (rotate labels 90°), y=log2FoldChange; title and labels required.

ROBUSTNESS:
- Coerce numeric; drop missing rows.

EXPORT:
- figures/<safe_title>.png (dpi=300, bbox_inches="tight")

OUTPUT:
- STRICT JSON ONLY.
"""


In [ ]:

schema_topup = """
{
  "datasets": {
    "top50_up": {
      "description": "Top 50 upregulated genes",
      "columns": ["Gene","log2FoldChange"]
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""

user_msg_topup = f"""
SCHEMA:
{schema_topup}

GOAL:
- Generate a Top‑50 Up bar/strip plot spec (STRICT JSON) using top50_up:
  X=gene, Y=log2FoldChange; rotate x labels 90°.

RESTRICTIONS:
- Use ONLY top50_up; define helpers inside the code as needed; no invented column names.

FORMAT:
- Return STRICT JSON array: [{{title,description,dependencies,code}}, ...].
"""


In [ ]:

import json, os
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
import re

resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_topup},
        {"role": "user",   "content": user_msg_topup}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content
plan_topup = json.loads(response_text)

sns.set_context("talk"); sns.set_palette("colorblind")

top50_up = top50_up.copy()
for c in ["log2FoldChange","padj"]:
    if c in top50_up.columns:
        top50_up[c] = pd.to_numeric(top50_up[c], errors="coerce")

local_env = {"top50_up": top50_up, "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os, "re":re}

local_env.update({"palette":"colorblind","colorblind":"colorblind","colorvblind":"colorblind","colourblind":"colorblind","cb_palette":"colorblind","re":"re"})

for fig in plan_topup:
    code = fig["code"]
    safe_title = fig.get("title","Top50_Up").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig,f,indent=2)
    exec(code, {}, local_env)


### Top‑50 Downregulated genes barplot

In [ ]:

system_msg_topdown = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs.

APPROVED DATAFRAME NAME:
- top50_down

PALETTE / STYLE:
- Define any palette INSIDE the code block if needed.

COLUMNS:
- top50_down: ["Gene","log2FoldChange"]

FIGURE:
- Barplot/strip for Top‑50 Down: x=gene (rotate labels 90°), y=log2FoldChange.

ROBUSTNESS:
- Coerce numerics; drop missing rows.

EXPORT:
- figures/<safe_title>.png (dpi=300, bbox_inches="tight")

OUTPUT:
- STRICT JSON ONLY.
"""


In [ ]:

schema_topdown = """
{
  "datasets": {
    "top50_down": {
      "description": "Top 50 downregulated genes",
      "columns": ["Gene","log2FoldChange"]
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""

user_msg_topdown = f"""
SCHEMA:
{schema_topdown}

GOAL:
- Generate a Top‑50 Down bar/strip plot spec (STRICT JSON) using top50_down:
  X=gene, Y=log2FoldChange; rotate x labels 90°.

RESTRICTIONS:
- Use ONLY top50_down; define helper variables INSIDE code if needed; do not invent columns.

FORMAT:
- Return STRICT JSON array: [{{title,description,dependencies,code}}, ...].
"""

In [ ]:

# === TOP-50 DOWN: execute plan and plot ===
import json, os
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_topdown},
        {"role": "user",   "content": user_msg_topdown}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content
plan_topdown = json.loads(response_text)

os.makedirs("figures", exist_ok=True)
sns.set_context("talk"); sns.set_palette("colorblind")

top50_down = top50_down.copy()
for c in ["log2FoldChange","padj"]:
    if c in top50_down.columns:
        top50_down[c] = pd.to_numeric(top50_down[c], errors="coerce")

local_env = {"top50_down": top50_down, "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os}

local_env.update({"palette":"colorblind","colorblind":"colorblind","colorvblind":"colorblind","colourblind":"colorblind","cb_palette":"colorblind"})

for fig in plan_topdown:
    code = fig["code"]
    safe_title = fig.get("title","Top50_Down").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig,f,indent=2)
    exec(code, {}, local_env)


### PCA Plot

In [ ]:

system_msg_pca = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs.

APPROVED DATAFRAME NAME:
- pca_df

PALETTE / STYLE:
- No external palette variables; define inside code if needed.

COLUMNS:
- pca_df: ["Sample","PC1","PC2","Treatment"]

FIGURE:
- PCA scatter: X=PC1, Y=PC2, hue=Treatment; add legend; title and axis labels required.

ROBUSTNESS:
- Coerce numerics; drop missing rows.

EXPORT:
- figures/<safe_title>.png (dpi=300, bbox_inches="tight")

OUTPUT:
- STRICT JSON ONLY.
"""


In [ ]:

schema_pca = """
{
  "datasets": {
    "pca_df": {
      "description": "PCA coordinates",
      "columns": ["Sample","PC1","PC2","Treatment"]
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""
user_msg_pca = f"""
SCHEMA:
{schema_pca}

GOAL:
- Generate a PCA scatter spec (STRICT JSON) using pca_df:
  X=PC1, Y=PC2, color by Treatment; include legend, title, axis labels.

RESTRICTIONS:
- Use ONLY pca_df; define helper variables INSIDE code if needed; do not invent columns.

FORMAT:
- Return STRICT JSON array: [{{title,description,dependencies,code}}, ...].
"""

In [ ]:

import json, os
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_pca},
        {"role": "user",   "content": user_msg_pca}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content
plan_pca = json.loads(response_text)

os.makedirs("figures", exist_ok=True)
sns.set_context("talk"); sns.set_palette("colorblind")

pca_df = pca_df.copy()
for c in ["PC1","PC2"]:
    pca_df[c] = pd.to_numeric(pca_df[c], errors="coerce")
pca_df["Treatment"] = pca_df["Treatment"].astype(str)

local_env = {"pca_df": pca_df, "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os}

local_env.update({"palette":"colorblind","colorblind":"colorblind","colorvblind":"colorblind","colourblind":"colorblind","cb_palette":"colorblind"})

for fig in plan_pca:
    code = fig["code"]
    safe_title = fig.get("title","PCA_scatter").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig,f,indent=2)
    exec(code, {}, local_env)


### Sample‑distance heatmap 

In [ ]:

system_msg_heatmap = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs.

APPROVED DATAFRAME NAME:
- dist_corr_df   (preferred)
- If not present, dist_euclid_df or dist_spear_df may be used (but use whatever is provided in the notebook).

STYLE / CLUSTERING:
- If clustermap is used, ensure scipy is available and the matrix is not empty or all-NaN.
- If clustering fails, fall back to a plain sns.heatmap with the same formatting.

FIGURE:
- Sample‑distance heatmap: square cells, thin white borders, viridis palette, colorbar, axis tick labels as sample IDs.

ROBUSTNESS:
- Ensure the matrix is square and indexes/columns are identical sets of sample IDs.
- Drop entirely NaN rows/columns; ensure finite values; if necessary, replace inf with NaN and fill NaN with column means.

EXPORT:
- figures/<safe_title>.png (dpi=300, bbox_inches="tight")

OUTPUT:
- STRICT JSON ONLY.
"""


In [ ]:

schema_heatmap = """
{
  "datasets": {
    "pca_df": {
      "description": "dist_corr_df",
      "columns": "n x n"
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""

user_msg_heatmap = f"""
SCHEMA:
{schema_heatmap}

GOAL:
- Generate a sample‑distance heatmap spec (STRICT JSON) from dist_corr_df:
  square cells, thin white borders, viridis palette, colorbar, axis tick labels.

RESTRICTIONS:
- Use ONLY dist_corr_df; define helper variables INSIDE code if needed; do not invent columns.

FORMAT:
- Return STRICT JSON array: [{{title,description,dependencies,code}}, ...].
"""


In [ ]:

# === SAMPLE-DISTANCE HEATMAP: execute plan and plot ===
import json, os
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_heatmap},
        {"role": "user",   "content": user_msg_heatmap}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content
plan_heatmap = json.loads(response_text)

os.makedirs("figures", exist_ok=True)
sns.set_context("talk"); sns.set_palette("colorblind")

# Keep symmetric intersection if needed

common = sorted(set(corr_df.index) & set(corr_df.columns))
corr_df = corr_df.loc[common, common]

local_env = {"dist_corr_df": corr_df, "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os}
local_env.update({"palette":"colorblind","colorblind":"colorblind","colorvblind":"colorblind","colourblind":"colorblind","cb_palette":"colorblind"})

for fig in plan_heatmap:
    code = fig["code"]
    safe_title = fig.get("title","Sample_distance_heatmap").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig,f,indent=2)
    exec(code, {}, local_env)


### Gene × Sample clustered heatmap (subset of top DE genes)

In [ ]:

system_msg_gxh = """
You are a senior bioinformatics visualization assistant.

Return a STRICT JSON array of figure specs.

APPROVED DATAFRAME NAME:
- gene_samples_df

ORIENTATION:
- Detect whether gene_samples_df is genes×samples or samples×genes.
- If needed, transpose for the chosen visualization.
- Subset ~50–200 genes (e.g., top |log2FoldChange| with padj<0.05 joined from res_shrunken if available; otherwise variance-based).

PALETTE / STYLE:
- No external palette variables; if needed, define inside the code.

ROBUSTNESS:
- Coerce numerics; handle NaN/inf; ensure ≥2 rows and ≥2 columns for clustering.
- If clustermap dendrogram fails (empty/NaN distances), fall back to sns.heatmap with the same formatting.

FIGURE:
- Clustered gene×sample heatmap with viridis palette, colorbar, labeled axes; rotate tick labels for readability.

EXPORT:
- figures/<safe_title>.png (dpi=300, bbox_inches="tight")

OUTPUT:
- STRICT JSON ONLY.
"""


In [ ]:

schema_gxh = """
{
  "datasets": {
    "gene_samples_df": {
      "description": "Gene × Sample expression matrix",
      "orientation": "rows = genes OR rows = samples"
    }
  },
  "output": {
    "dir": "figures",
    "export": { "png": true },
    "style": { "context": "talk", "palette": "colorblind" }
  }
}
"""
user_msg_gxh = f"""
SCHEMA:
{schema_gxh}

GOAL:
- Generate a clustered gene×sample heatmap spec (STRICT JSON) from gene_samples_df:
  detect orientation, subset to ~50–200 genes, viridis palette, colorbar, labeled axes.

RESTRICTIONS:
- Use ONLY gene_samples_df (and res_shrunken if you choose to select genes by DE stats); define helper variables INSIDE code; do not invent columns.

FORMAT:
- Return STRICT JSON array: [{{title,description,dependencies,code}}, ...].
"""


In [ ]:

# === GENE×SAMPLE CLUSTERED HEATMAP: execute plan and plot ===
import json, os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

resp = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": system_msg_gxh},
        {"role": "user",   "content": user_msg_gxh}
    ],
    temperature=0.2
)
response_text = resp.choices[0].message.content
plan_gxh = json.loads(response_text)

os.makedirs("figures", exist_ok=True)
sns.set_context("talk"); sns.set_palette("colorblind")

# orientation detection + numeric coercion
gene_samples_df = gene_samples_df.copy()
# build sample_ids from metadata (used by many generated snippets)
sample_ids = list(samples_meta["ID"].astype(str))
gene_samples_df.columns = gene_samples_df.columns.astype(str)
gene_samples_df.index = gene_samples_df.index.astype(str)


def to_samples_by_genes(df, sample_ids):
    # if samples in index → OK; if samples in columns → transpose
    if set(sample_ids).issubset(set(df.index)):
        out = df.copy()
    elif set(sample_ids).issubset(set(df.columns)):
        out = df.T.copy()
    else:
        out = df.copy()
    # numeric coercion
    out = out.apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
    # drop columns entirely NaN; fill partial NaNs with column mean
    all_nan_cols = [c for c in out.columns if out[c].isna().all()]
    if all_nan_cols:
        out = out.drop(columns=all_nan_cols)
    out = out.fillna(out.mean())
    return out

expr_sxg = to_samples_by_genes(gene_samples_df, sample_ids)

local_env = {
    "gene_samples_df": gene_samples_df,  # raw
    "expr_sxg": expr_sxg,                # rows=samples, cols=genes
    "samples_meta": samples_meta,
    "sample_ids": sample_ids,
    "sns": sns, "plt": plt, "np": np, "pd": pd, "os": os
}

local_env.update({"palette":"colorblind","colorblind":"colorblind","colorvblind":"colorblind","colourblind":"colorblind","cb_palette":"colorblind"})

for fig in plan_gxh:
    code = fig["code"]
    safe_title = fig.get("title","GeneSample_clustered_heatmap").replace(" ","_")
    with open(f"figures/{safe_title}_plan.json","w") as f: json.dump(fig,f,indent=2)
    exec(code, local_env)


### Conclusion

- Generate figures using Azure Open AI. Key skills you learned were to:
   - Azure Environment Setup for AI
   - What is a prompt?
   - Four layers of prompt
   - AI generated RNA-seq figures

### Cleanup

Warning: Dont forget to delete the resources we just made to avoid accruing additional costs!